# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 8 · Parallel Swarm — Multi-Agent Fan-Out & Alignment Merge

**Phase B · Orchestration** &nbsp;|&nbsp; **Goal:** Parallelise the per-unit work and select/merge the best

---

**Description**

Demonstrates genuine parallel agent execution (fan-out) via a thread pool,
followed by an explicit, schema-driven merge and conflict resolution stage (fan-in).
In the Research Grant System:
1. Narrative sections ('abstract', 'background', 'methodology', 'budget') are authored
   concurrently by parallel drafting workers.
2. An explicit Merge stage evaluates outputs against the locked plan constraints.
3. A Critique stage checks for inter-agent conflicts (e.g., equipment proposed by
   Methodology exceeding the ceiling enforced by the Budget agent).
4. A Conflict Resolution stage resolves tensions using deterministic priority rules.
5. Wall-clock concurrency vs sequential estimate is benchmarked to report speedup.

**Assignments included**

- Assignment 1: Add a fifth role / section ('data_management_agent') proposing secure storage cost.
- Assignment 2: Distributed critique where each specialist independently audits the merged plan.

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab08_parallel_swarm.py`](./scripts/lab08_parallel_swarm.py).*

### Cell 1 · Imports and fixtures

In [1]:
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Any, List

### Cell 2 · `literature_scout_agent` (function)

In [2]:
# ==============================================================================
# Role Specialist Workers (Genuinely Independent)
# ==============================================================================

def literature_scout_agent(goal: str) -> Dict[str, Any]:
    time.sleep(0.05)  # simulate processing
    return {
        "role": "literature_scout",
        "output": {
            "key": "background",
            "text": "State-of-the-art review: Federated learning reduces central hospital data exposure. 4 papers cited.",
            "citations": ["lit-001", "lit-002", "lit-003", "lit-004"],
            "word_count": 140,
        }
    }

### Cell 3 · `methodology_agent` (function)

In [3]:
def methodology_agent(goal: str) -> Dict[str, Any]:
    time.sleep(0.06)
    return {
        "role": "methodology_specialist",
        "output": {
            "key": "methodology",
            "text": "Four clinical screening sites will run edge inference on non-mydriatic fundus cameras with local training.",
            "hardware_requested": "High-Performance GPU Cluster (INR 3,500,000)",
            "word_count": 175,
        }
    }

### Cell 4 · `budget_agent` (function)

In [4]:
def budget_agent(goal: str) -> Dict[str, Any]:
    time.sleep(0.04)
    return {
        "role": "budget_planner",
        "output": {
            "key": "budget",
            "equipment_ceiling": 2500000,  # 25 Lakh ceiling under SERB guidelines
            "total_budget_cap": 6000000,
            "proposed_direct": 5000000,
            "word_count": 95,
        }
    }

### Cell 5 · `compliance_agent` (function)

In [5]:
def compliance_agent(goal: str) -> Dict[str, Any]:
    time.sleep(0.05)
    return {
        "role": "compliance_officer",
        "output": {
            "key": "compliance",
            "ethics_clearance_required": True,
            "dpdp_plan_required": True,
            "word_count": 85,
        }
    }

ROLES = {
    "literature": literature_scout_agent,
    "methodology": methodology_agent,
    "budget": budget_agent,
    "compliance": compliance_agent,
}

### Cell 6 · `run_grant_swarm` (function)

In [6]:
# ==============================================================================
# Swarm Execution & Plan-Alignment Merge
# ==============================================================================

def run_grant_swarm(goal: str, active_roles: Dict) -> Dict[str, Any]:
    """Runs all specialist agents concurrently using ThreadPoolExecutor."""
    start_time = time.time()
    outputs = {}

    with ThreadPoolExecutor(max_workers=len(active_roles)) as executor:
        futures = {name: executor.submit(fn, goal) for name, fn in active_roles.items()}
        for name, future in futures.items():
            outputs[name] = future.result()

    wall_clock = time.time() - start_time
    return {"outputs": outputs, "wall_clock_sec": wall_clock}

### Cell 7 · `merge_by_plan` (function)

In [7]:
def merge_by_plan(swarm_results: Dict[str, Any], target_sections: List[str]) -> Dict[str, Any]:
    """
    Combines independent outputs into a single draft proposal package.
    Audits coverage against target units in the locked plan.
    """
    outputs = swarm_results["outputs"]
    merged_sections = {}
    coverage = []

    for sec in target_sections:
        # Find which role produced this section
        found = None
        for role_name, res in outputs.items():
            if res["output"].get("key") == sec:
                found = res["output"]
                break

        if found:
            merged_sections[sec] = found
            coverage.append({"section": sec, "status": "covered", "words": found.get("word_count", 0)})
        else:
            coverage.append({"section": sec, "status": "missing", "words": 0})

    return {
        "sections": merged_sections,
        "coverage": coverage,
        "all_covered": all(c["status"] == "covered" for c in coverage),
    }

### Cell 8 · `critique_draft` (function)

In [8]:
def critique_draft(merged_draft: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Audits cross-agent constraints and surfaces explicit conflicts.
    """
    conflicts = []
    secs = merged_draft["sections"]

    # Tension 1: Methodology hardware request vs Budget agency ceiling
    if "methodology" in secs and "budget" in secs:
        meth = secs["methodology"]
        bgt = secs["budget"]
        if "GPU Cluster (INR 3,500,000)" in meth.get("hardware_requested", ""):
            requested_cost = 3500000
            allowed_ceiling = bgt.get("equipment_ceiling", 0)
            if requested_cost > allowed_ceiling:
                conflicts.append({
                    "conflict_id": "CONF-001",
                    "conflict": "Equipment cost in methodology exceeds agency ceiling",
                    "roles_involved": ["methodology_specialist", "budget_planner"],
                    "requested_value": requested_cost,
                    "ceiling_value": allowed_ceiling,
                })

    return conflicts

### Cell 9 · `resolve_conflict` (function)

In [9]:
def resolve_conflict(conflict: Dict[str, Any], priority_rule: str = "agency_ceiling_over_spec") -> Dict[str, Any]:
    """
    Resolves identified tensions using documented governance rules.
    """
    if conflict["conflict_id"] == "CONF-001":
        return {
            "rule_applied": priority_rule,
            "resolution": (
                f"Cap equipment at INR {conflict['ceiling_value']:,}; "
                f"direct methodology agent to utilize institutional shared cloud compute for remaining capacity."
            ),
            "revised_equipment_cost": conflict["ceiling_value"],
        }
    return {"rule_applied": "none", "resolution": "no rule matched"}

### Cell 10 · `data_management_agent` (function)

In [10]:
# ==============================================================================
# Graded Assignments
# ==============================================================================

def data_management_agent(goal: str) -> Dict[str, Any]:
    """Assignment 1 Role: Proposes data storage and DPDP compliance infrastructure."""
    time.sleep(0.04)
    return {
        "role": "data_management_officer",
        "output": {
            "key": "data_management",
            "storage_cost": 800000,
            "text": "Data Management Plan: Encrypted on-premise vault with biometric access logging.",
            "word_count": 115,
        }
    }

### Cell 11 · `run_assignment_1_fifth_role` (function)

In [11]:
def run_assignment_1_fifth_role():
    """
    Assignment 1 -- Add a Fifth Role (Data Management):
    Extends the swarm with a data management officer, runs in parallel,
    and incorporates it into the merged draft.
    """
    print("\n--- [Assignment 1] Add Fifth Role: Data Management ---")
    roles_5 = dict(ROLES)
    roles_5["data_management"] = data_management_agent

    swarm_res = run_grant_swarm("Plan SERB CRG Grant", roles_5)
    merged = merge_by_plan(swarm_res, ["background", "methodology", "budget", "compliance", "data_management"])
    print(f"5-Agent Swarm completed in {swarm_res['wall_clock_sec']:.3f} seconds.")
    print("Coverage report across 5 units:")
    for c in merged["coverage"]:
        print(f"  Unit '{c['section']}': {c['status']} ({c['words']} words)")
    assert merged["all_covered"], "All 5 units should be covered!"

### Cell 12 · `run_assignment_2_distributed_critique` (function)

In [12]:
def run_assignment_2_distributed_critique(merged_draft: Dict[str, Any]):
    """
    Assignment 2 -- Distributed Critique:
    Instead of a single monolith function, each specialist independently audits
    the merged plan against their domain rules and logs objections.
    """
    print("\n--- [Assignment 2] Distributed Specialist Critique ---")
    objections = []

    # 1. Budget Specialist checks equipment ceiling
    bgt = merged_draft["sections"].get("budget", {})
    meth = merged_draft["sections"].get("methodology", {})
    if "3,500,000" in meth.get("hardware_requested", "") and bgt.get("equipment_ceiling", 0) < 3500000:
        objections.append({
            "auditing_role": "budget_planner",
            "severity": "blocking",
            "objection": f"Methodology requested hardware of 35L exceeding budget cap of {bgt['equipment_ceiling']} INR.",
        })

    # 2. Compliance Specialist checks ethics requirement in methodology
    comp = merged_draft["sections"].get("compliance", {})
    if comp.get("ethics_clearance_required") and "fundus" in meth.get("text", ""):
        objections.append({
            "auditing_role": "compliance_officer",
            "severity": "advisory",
            "objection": "Methodology involves human clinical fundus imaging; verified ethics document must be attached.",
        })

    for obj in objections:
        print(f"  [{obj['auditing_role']}] ({obj['severity'].upper()}): {obj['objection']}")
    assert len(objections) >= 2, "Expected multiple independent objections!"

### Cell 13 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [13]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 8: Parallel Swarm -- Multi-Agent Fan-Out & Alignment Merge")
    print("======================================================================")

    # 1. Execute concurrent swarm
    print("Launching 4 specialist agents in parallel via ThreadPoolExecutor...")
    swarm_result = run_grant_swarm("Draft SERB CRG Grant for Retinopathy Screening", ROLES)
    print(f"Execution complete. Wall-clock time: {swarm_result['wall_clock_sec']:.3f}s")
    for role, r in swarm_result["outputs"].items():
        print(f"  [{role}] produced section '{r['output'].get('key')}' ({r['output'].get('word_count')} words)")

    # 2. Alignment Merge against target units
    target_units = ["background", "methodology", "budget", "compliance"]
    merged_plan = merge_by_plan(swarm_result, target_units)
    print("\n=== Plan Alignment Merge ===")
    print(f"All Required Units Covered: {merged_plan['all_covered']}")

    # 3. Conflict Critique & Resolution
    print("\n=== Cross-Agent Conflict Critique ===")
    conflicts = critique_draft(merged_plan)
    conflict_log = []
    for c in conflicts:
        print(f"Detected: {c['conflict']} (between {c['roles_involved']})")
        res = resolve_conflict(c)
        conflict_log.append({**c, **res})
        print(f"  -> Resolution applied: {res['resolution']}")

    # 4. Assignments
    run_assignment_1_fifth_role()
    run_assignment_2_distributed_critique(merged_plan)

    print("\n=== Final Conflict Log (Swarm Report Artifact) ===")
    for log_entry in conflict_log:
        print(f"Conflict: {log_entry['conflict']}")
        print(f"  Rule: {log_entry['rule_applied']}")
        print(f"  Action: {log_entry['resolution']}")

Lab 8: Parallel Swarm -- Multi-Agent Fan-Out & Alignment Merge
Launching 4 specialist agents in parallel via ThreadPoolExecutor...
Execution complete. Wall-clock time: 0.061s
  [literature] produced section 'background' (140 words)
  [methodology] produced section 'methodology' (175 words)
  [budget] produced section 'budget' (95 words)
  [compliance] produced section 'compliance' (85 words)

=== Plan Alignment Merge ===
All Required Units Covered: True

=== Cross-Agent Conflict Critique ===
Detected: Equipment cost in methodology exceeds agency ceiling (between ['methodology_specialist', 'budget_planner'])
  -> Resolution applied: Cap equipment at INR 2,500,000; direct methodology agent to utilize institutional shared cloud compute for remaining capacity.

--- [Assignment 1] Add Fifth Role: Data Management ---
5-Agent Swarm completed in 0.061 seconds.
Coverage report across 5 units:
  Unit 'background': covered (140 words)
  Unit 'methodology': covered (175 words)
  Unit 'budget': cov

### Cell 14 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab8_artifacts/` (committed under `Course-Project/evidence/`).

In [14]:
import json
import re
from pathlib import Path

LAB = 8
CHECKS = [('Four specialists fan out in a real thread pool', 'Launching 4 specialist agents in parallel via ThreadPoolExecutor', 1),
    ('Merge covers every unit required by the plan', 'All Required Units Covered: True', 1),
    ('Cross-agent conflict is detected and resolved by rule', 'Resolution applied', 1),
    ('Fifth role (data management) joins the swarm', "Unit 'data_management': covered", 1),
    ('Specialist critique raises a blocking finding', '\\(BLOCKING\\)', 1),
    ('Specialist critique raises an advisory finding', '\\(ADVISORY\\)', 1)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Four specialists fan out in a real thread pool
[PASS]  5. Merge covers every unit required by the plan
[PASS]  6. Cross-agent conflict is detected and resolved by rule
[PASS]  7. Fifth role (data management) joins the swarm
[PASS]  8. Specialist critique raises a blocking finding
[PASS]  9. Specialist critique raises an advisory finding
[PASS] 10. No unhandled exception in the run

Lab 8: 10/10 acceptance checks passed
Artifacts written to lab8_artifacts
